<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 13


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Inventory в C#, который будет представлять информацию о наличии товаров на складе. На основе этого класса разработать 2-3 производных класса, демонстрирующих принципы наследования и полиморфизма. В каждом из классов должны быть реализованы новые атрибуты и методы, а также переопределены некоторые методы базового класса для демонстрации полиморфизма.

Требования к базовому классу Inventory:

Атрибуты: ID склада (WarehouseId), Название склада (WarehouseName), Общий объем хранения (StorageCapacity).

Методы:

• GetStorageStatus(): метод для получения статуса доступного пространства на складе.

• AddItem(Item item): метод для добавления товара на склад.

• RemoveItem(Item item): метод для удаления товара со склада.

Требования к производным классам:

ПерсональныйСклад (PersonalInventory): Должен содержать дополнительные атрибуты, такие как Владелец склада (OwnerName). Метод GetStorageStatus() должен быть переопределен для отображения информации о владельце склада вместе с статусом хранения.
ГрупповойСклад (GroupInventory): Должен содержать дополнительные атрибуты, такие как Группа товаров (ProductGroup). Метод AddItem() должен быть переопределен для добавления информации о группе товаров при добавлении нового товара.
АвтоматизированныйСклад (AutomatedInventory) (если требуется третий класс): Должен содержать дополнительные атрибуты, такие как Автоматизация уровня (AutomationLevel). Метод RemoveItem() должен быть переопределен для добавления информации о уровне автоматизации при удалении товара.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [2]:
using System;
using System.Reflection;
using System.Collections.Generic;

public class Item
{
    public string ItemType; 

    public Item(string _ItemType)
    {
        ItemType = _ItemType;
    }

}

// Добавление делегата для уведомлений об изменении состояния склада
public delegate void InventoryUpdateHandler(string message);

public abstract class BaseInventory
{
    public int _WareHouseId;  
    public string _WareHouseName;
    public int _StorageCapacity;
    public List<Item> items; // список добавленных товаров

    // Новое событие
    public event InventoryUpdateHandler InventoryUpdate; // Событие InventoryUpdate основано на делегате InventoryUpdateHandler

    public int WareHouseId
    {
        get{return _WareHouseId;}
        set{_WareHouseId = value;}
    }

    public string WareHouseName
    {
        get{return _WareHouseName;}
        set{_WareHouseName = value;}
    }
    
    public int StorageCapacity
    {
        get{return _StorageCapacity;}
        set
        {
            if(value >= 0)
            {
                _StorageCapacity = value;
            } 
            else
                throw new ArgumentOutOfRangeException("Объем хранения не может быть отрицательным!");
        }
    }

    public BaseInventory(int warehouseid, string warehousename, int storagecapacity)
    {
        WareHouseId = warehouseid;
        WareHouseName = warehousename;
        StorageCapacity = storagecapacity;
        items = new List<Item>();
    }

    // Вызов события InventoryUpdate если есть подписчики(методы, которые зарегистрированы на событие)
    protected void NotifyUpdate(string message)
    {
        InventoryUpdate?.Invoke(message);
    }

}

public class Inventory : BaseInventory // пример простого наследования
{
    public DateTime LastUpdated {get; set;} // добавляем дату последнего обновления
    public Inventory(int warehouseId, string warehouseName, int storageCapacity)
        : base(warehouseId, warehouseName, storageCapacity)
    { 

    }


    public virtual void GetStorageStatus()
    {
        Console.WriteLine($"На Складе '{WareHouseName}' с ID:{WareHouseId} доступно места: {StorageCapacity - items.Count}");
    }

    public virtual void AddItem(Item item)
    {
        if(items.Count <= StorageCapacity) // проверяем, чтобы количество товаров не превосходило общий объем хранения
        {
            items.Add(item);
            Console.WriteLine($"Товар '{item.ItemType}' добавлен на склад '{WareHouseName}'");
            NotifyUpdate($"Добавлен товар '{item.ItemType}' на склад '{WareHouseName}'");  // 
        }
        else
        {
            Console.WriteLine($"Не удалось добавить товар '{item.ItemType}', так как недостаточно места.");
        }
        LastUpdated = DateTime.Now;
        Console.WriteLine($"Инвентарь обновлен! Дата и время: {LastUpdated}");

    }

    public virtual void RemoveItem(Item item)
    {
        Console.WriteLine($"Товар '{item.ItemType}' удален со склада '{WareHouseName}'");
        NotifyUpdate($"Удалён товар '{item.ItemType}' со склада '{WareHouseName}'"); // Метод с оповещением
        LastUpdated = DateTime.Now;
        Console.WriteLine($"Инвентарь обновлен! Дата и время: {LastUpdated}");
    }

    public virtual void SellStorage(int storage_price)
    {
        Console.WriteLine($"Стоимость склада: {storage_price}");
    }

    public virtual void GenericTest()
    {
        Console.WriteLine("Тест generic класса в Inventory");
    }

}


public interface ISellable
{
    void SellStorage(string newOwner, string price);   
}

public class PersonalInventory : Inventory, ISellable
{
    private string _OwnerName;
    private string _ContactInfo; // добавляем поле с контактной информацией


    public string ContactInfo // добавляем Свойство с контактной информацией
    {
        get{return _ContactInfo;}
        set{_ContactInfo = value;}
    }

    public string OwnerName
    {
        get{return _OwnerName;}
        set{_OwnerName = value;}
    }
    
    public PersonalInventory(int warehouseid, string warehousename, int storagecapacity, string ownername, string contactinfo)
        : base(warehouseid, warehousename, storagecapacity)
    {
        OwnerName = ownername;
        ContactInfo = contactinfo; // объвляем в конструкторе   
    }

    //Перегрузка метода

    public virtual void GetStorageStatus(Item item)
    {
         Console.WriteLine($"На Складе '{WareHouseName}' с ID:{WareHouseId} c владельцем: {OwnerName} хранятся товары типа: '{item.ItemType}'. Доступно места: {StorageCapacity - items.Count}.");
    }

    public override void SellStorage(int storage_price)
    {
        Console.WriteLine($"Стоимость склада: {storage_price} рублей");
    }

    void ISellable.SellStorage(string newOwner, string price) // явная реализация интерфейса
    {
        Console.WriteLine($"{newOwner} купил склад у {OwnerName} за {price} рублей.");
    }

    public void GetOwnerContactInfo() // метод с добавлением информации о владельце
    {
        Console.WriteLine($"Контакты {OwnerName}: {ContactInfo}.");
    }

    public override void GenericTest()
    {
        Console.WriteLine($"Id инвентаря PersonalInventory {WareHouseId}");
    }


}

public class GroupInventory : Inventory
{
    private string _ProductGroup;

    private int _ItemCount; // поле с количеством товара

    public int ItemCount
    {
        get{ return _ItemCount;}
        set{_ItemCount = value;}
    }

    public string ProductGroup
    {
        get{return _ProductGroup;}
        set{_ProductGroup = value;}
    }


    public GroupInventory(int warehouseid, string warehousename, int storagecapacity, string productgroup)
    : base(warehouseid, warehousename, storagecapacity)
    {
        ProductGroup = productgroup;
        ItemCount = 0; // Изначально товара нет 
    }

    public override void AddItem(Item item)
    {
        if (items.Count < StorageCapacity)
        {
            items.Add(item);
            ItemCount++;
            Console.WriteLine($"Товар '{item.ItemType}' добавлен в группу '{ProductGroup}' на склад '{WareHouseName}', количество товара {ItemCount}.");
            LastUpdated = DateTime.Now;
            NotifyUpdate($"Добавлен товар '{item.ItemType}' на склад '{WareHouseName}'");
            Console.WriteLine($"Инвентарь обновлен! Дата и время: {LastUpdated}");        
        }
        else
        {
            Console.WriteLine($"Не удалось добавить товар '{item.ItemType}' в группу '{ProductGroup}' на складе '{WareHouseName}': нет свободного места.");
        }

    }

    public void StorageLocation(PersonalInventory myPersonalInventory, string City)
    {
        Console.WriteLine($"Склад '{myPersonalInventory.WareHouseName}' с id: {myPersonalInventory.WareHouseId} и владельцем {myPersonalInventory.OwnerName} находится в городе {City}.");
    }

    public void GetItemCount()
    {
        Console.WriteLine(ItemCount);
    }


    public override void GenericTest()
    {
        Console.WriteLine($"Id инвентаря GroupInventory {WareHouseId}");
    }

}

public class AutomatedInventory : Inventory
{
    public string AutomationLevel {get;set;}
    private bool IsFullyAutomated {get; set;} // Свойство полностью автоматизированного склада


    public AutomatedInventory( int _WareHouseId, string _WareHouseName, int _StorageCapacity, string _AutomationLevel,  bool isFullyAutomated)
    : base(_WareHouseId, _WareHouseName, _StorageCapacity)
    {
        AutomationLevel = _AutomationLevel;
        IsFullyAutomated = isFullyAutomated;
    }

    public override void AddItem(Item item)
    {
        base.AddItem(item);
        NotifyUpdate($"[Автоматизировано] Товар '{item.ItemType}' добавлен на автоматизированный склад '{WareHouseName}'.");
    }

    public override void RemoveItem(Item item)
    {
        base.RemoveItem(item);// обращение к базовому классу
        NotifyUpdate($"[Автоматизировано] Товар '{item.ItemType}' удален с автоматизированного склада '{WareHouseName}'.");
        Console.WriteLine($"Уровень автоматизации '{AutomationLevel}' при удалении товара '{item.ItemType}' со склада '{WareHouseName}'.");
    }

    public void PerfomeAutomatedProcess() // Выполнение автоматизации процесса
    {
        if(IsFullyAutomated == true)
        {
            Console.WriteLine("Полностью автоматизированный процесс!");
        }
        else
        {
            Console.WriteLine("Процесс автоматизирован не полностью!");
        }
    }

    public override void GenericTest()
    {
        Console.WriteLine($"Id инвентаря AutomatedInventory {WareHouseId}");
    }
    
}

public class InventoryManager
{
    // Коллекция объектов типа Inventory
    private List<Inventory> inventories = new List<Inventory>();

    public void AddInventory(Inventory inventory)
    {
        inventories.Add(inventory);
        Console.WriteLine($"Инвентарь '{inventory.WareHouseName}' добавлен в систему.");
    }

    public void DisplayAllInventories()
    {
        foreach (var inventory in inventories)
        {
            Console.WriteLine($"Склад: {inventory.WareHouseName}, ID: {inventory.WareHouseId}, Доступное место: {inventory.StorageCapacity - inventory.items.Count}");
        }
    }

    public IEnumerable<Inventory> FilterInventoriesByCapacity(int minCapacity)
    {
        return inventories.FindAll(с => с.StorageCapacity >= minCapacity);
    }
}

// реализация генерического класса InventoryCollection
public class InventoryCollection<T> where T : Inventory
{
    private List<T> _inventory = new List<T>();

    public void Add(T inventory)
    {
        _inventory.Add(inventory);
    }

    public void DisplayInventory()
    {
        foreach (var inventory in _inventory)
        {
            // Пример вызова метода для каждого элемента, если метод известен.
            inventory.GenericTest();
        }
    }
}

Inventory myInventory = new Inventory(1, "Книги", 10);
PersonalInventory myPersonalInventory = new PersonalInventory(1, "Мебель", 40, "Алик Мирзоев", "sugrovskiyN@mail.com");
PersonalInventory myPersonalInventory2 = new PersonalInventory(8, "Мебель", 40, "Алик Мирзоев", "sugrovskiyN@mail.com");

GroupInventory myGroupInventory = new GroupInventory(2, "Электроника", 80, "Гаджеты");
AutomatedInventory myAutomatedInventory = new AutomatedInventory(3, "Одежда", 55, "Высокий", true);

// Менеджер инвентаря
InventoryManager manager = new InventoryManager();

// Подписка на события
myPersonalInventory.InventoryUpdate += message => Console.WriteLine($"[Уведомление]: {message}");
myGroupInventory.InventoryUpdate += message => Console.WriteLine($"[Уведомление]: {message}");
myAutomatedInventory.InventoryUpdate += message => Console.WriteLine($"[Уведомление]: {message}");
  
// Добавляем инвентарь в менеджер
manager.AddInventory(myPersonalInventory);
manager.AddInventory(myAutomatedInventory);
manager.AddInventory(myGroupInventory);
Console.WriteLine();


// Добавляем товары
myPersonalInventory.AddItem(new Item("Кухонный гарнитур"));
Console.WriteLine();
myGroupInventory.AddItem(new Item("Телефон"));
myGroupInventory.AddItem(new Item("Ноутбук"));
Console.WriteLine();
myAutomatedInventory.AddItem(new Item("Футболка"));
Console.WriteLine();


// Удаляем товары
myGroupInventory.RemoveItem(new Item("Телефон"));
Console.WriteLine();


// Отображаем все склады
Console.WriteLine("\nВсе склады:");
manager.DisplayAllInventories();

// Фильтруем склады
Console.WriteLine("\nСклады с вместимостью >= 50:");
var filteredInventories = manager.FilterInventoriesByCapacity(50);
foreach (var inv in filteredInventories)
{
    Console.WriteLine($"- {inv.WareHouseName}");
}


//InventoryCollection<Inventory> inventory = new InventoryCollection<Inventory>();
//inventory.Add(new PersonalInventory(1, "Мебель", 10, "Овсепян Арам", "github_090506@mail.ru"));
//inventory.Add(new GroupInventory(2, "Электроника", 3, "Гаджеты"));
//inventory.Add(new AutomatedInventory(3, "Одежда", 15, "Высокий", true));
//inventory.DisplayInventory();
//Console.WriteLine();

//myPersonalInventory.GetStorageStatus(new Item("Диваны"));
//myPersonalInventory.AddItem(new Item("Книга 1"));
//myPersonalInventory.RemoveItem(new Item("Книга 1"));
//myPersonalInventory.GetOwnerContactInfo(); 
//myPersonalInventory.SellStorage(2200000);
//myPersonalInventory.SellStorage("Гурзан Владислав", "три миллиона"); // реализация новых методов
//ISellable _PersonalInventory = new PersonalInventory(1, "Мебель", 10, "Рябцев Никита", "github_090506@mail.ru"); // вызов явной реализации интерфейса
//_PersonalInventory.SellStorage("Гурзан Владислав", "два миллиона");
//Console.WriteLine();

//myGroupInventory.GetStorageStatus();
//myGroupInventory.AddItem(new Item("Iphone 15"));
//myGroupInventory.AddItem(new Item("Iphone 16"));
//myGroupInventory.RemoveItem(new Item("Телевизор"));
//myGroupInventory.StorageLocation(myPersonalInventory, "Алмата"); // взаимодествие объектов
//myGroupInventory.GetItemCount();
//Console.WriteLine();

//myAutomatedInventory.GetStorageStatus();
//myAutomatedInventory.AddItem(new Item("Футболка"));
//myAutomatedInventory.RemoveItem(new Item("Куртка"));
//yAutomatedInventory.PerfomeAutomatedProcess();

Инвентарь 'Мебель' добавлен в систему.
Инвентарь 'Одежда' добавлен в систему.
Инвентарь 'Электроника' добавлен в систему.

Товар 'Кухонный гарнитур' добавлен на склад 'Мебель'
[Уведомление]: Добавлен товар 'Кухонный гарнитур' на склад 'Мебель'
Инвентарь обновлен! Дата и время: 11/16/2025 1:57:21 PM

Товар 'Телефон' добавлен в группу 'Гаджеты' на склад 'Электроника', количество товара 1.
[Уведомление]: Добавлен товар 'Телефон' на склад 'Электроника'
Инвентарь обновлен! Дата и время: 11/16/2025 1:57:21 PM
Товар 'Ноутбук' добавлен в группу 'Гаджеты' на склад 'Электроника', количество товара 2.
[Уведомление]: Добавлен товар 'Ноутбук' на склад 'Электроника'
Инвентарь обновлен! Дата и время: 11/16/2025 1:57:21 PM

Товар 'Футболка' добавлен на склад 'Одежда'
[Уведомление]: Добавлен товар 'Футболка' на склад 'Одежда'
Инвентарь обновлен! Дата и время: 11/16/2025 1:57:21 PM
[Уведомление]: [Автоматизировано] Товар 'Футболка' добавлен на автоматизированный склад 'Одежда'.

Товар 'Телефон' удален с